In [1]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
import pickle
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score

c:\Projects\vlr-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
splits = joblib.load("splits/6_windows_all_features_knn.pkl")

In [3]:
splits[0]['X_train_raw'].shape

(1380, 1063)

In [4]:
from sklearn.impute import KNNImputer, SimpleImputer

feature_names = splits[0]["X_train_raw"].columns

for split in splits:

    # imputer = SimpleImputer(strategy="median")
    imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")

    split["X_train_raw"] = pd.DataFrame(imputer.fit_transform(split["X_train_raw"]), columns=feature_names)
    split["X_val_raw"] = pd.DataFrame(imputer.transform(split["X_val_raw"]), columns=feature_names)
    split["X_test_raw"] = pd.DataFrame(imputer.transform(split["X_test_raw"]), columns=feature_names)


In [5]:
def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi

In [6]:
# def make_objective(X_train, y_train, X_val, y_val):
#     def objective(trial):
#         C = trial.suggest_float('C', 1e-5, 10, log=True)
#         l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1)

#         all_val_true, all_val_proba = [], []

#         scaler = StandardScaler()
#         X_train_scaled = scaler.fit_transform(X_train)
#         X_val_scaled   = scaler.transform(X_val)

#         selector = VarianceThreshold(threshold=0.01)
#         X_train_sel = selector.fit_transform(X_train_scaled)
#         X_val_sel   = selector.transform(X_val_scaled)

#         model = LogisticRegression(C=C, l1_ratio=l1_ratio, solver='saga', max_iter=5000)
#         model.fit(X_train_sel, y_train)

#         val_proba = model.predict_proba(X_val_sel)[:, 1]
#         all_val_true.append(y_val); all_val_proba.append(val_proba)
#         logloss = log_loss(all_val_true, all_val_proba)
#         return logloss

#     return objective

In [7]:
from sklearn.model_selection import StratifiedKFold

def make_objective(X_train, y_train, n_inner_folds=5, seed=0):
    def objective(trial):
        C = trial.suggest_float('C', 1e-3, 10, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

        skf = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=seed)
        fold_losses = []

        for train_idx, inner_val_idx in skf.split(X_train, y_train):
            X_fold_train, X_fold_val = X_train[train_idx], X_train[inner_val_idx]

            scaler = StandardScaler()
            X_fold_train_scaled = scaler.fit_transform(X_fold_train)
            X_fold_val_scaled = scaler.transform(X_fold_val)

            selector = VarianceThreshold(threshold=0.01)
            X_fold_train_sel = selector.fit_transform(X_fold_train_scaled)
            X_fold_val_sel = selector.transform(X_fold_val_scaled)

            model = LogisticRegression(C=C, l1_ratio=l1_ratio, solver="saga", max_iter=5000)
            model.fit(X_fold_train_sel, y_train[train_idx])
            proba = model.predict_proba(X_fold_val_sel)[:, 1]
            fold_losses.append(log_loss(y_train[inner_val_idx], proba, labels=[0, 1]))

        return np.mean(fold_losses)

    return objective

In [ ]:
scores = []
parameters = []

for split in splits:
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    # objective = make_objective(X_train, y_train, X_val, y_val)
    objective = make_objective(X_train.values, y_train.values)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=500, n_jobs=-1)
    params = study.best_params
    parameters.append(params)
    print("Best parameters:", params)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel   = selector.transform(X_val_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_pred = model.predict(X_val_sel)
    scores.append(roc_auc_score(y_val, val_pred))

print(f"Scores: {scores}")
print(f"Parameters: {parameters}")
print(f"Mean logloss: {np.mean(scores)}")

[I 2026-07-23 10:48:36,142] A new study created in memory with name: no-name-bd105e85-463d-4005-8fbc-2f6697d003de
[I 2026-07-23 10:48:55,948] Trial 12 finished with value: 0.6580360768990046 and parameters: {'C': 0.012046486985287404, 'l1_ratio': 0.8016505550588144}. Best is trial 12 with value: 0.6580360768990046.
[I 2026-07-23 10:49:01,397] Trial 14 finished with value: 0.6564192250589802 and parameters: {'C': 0.014902699693400728, 'l1_ratio': 0.9516739051409029}. Best is trial 14 with value: 0.6564192250589802.
[I 2026-07-23 10:49:02,143] Trial 6 finished with value: 0.614235508978448 and parameters: {'C': 0.02404502155159138, 'l1_ratio': 0.046940149125011765}. Best is trial 6 with value: 0.614235508978448.
[I 2026-07-23 10:49:13,486] Trial 17 finished with value: 0.6361486818748784 and parameters: {'C': 0.009600134995728527, 'l1_ratio': 0.29303521521904985}. Best is trial 6 with value: 0.614235508978448.
[I 2026-07-23 10:49:18,661] Trial 18 finished with value: 0.6302706119290211 a

Best parameters: {'C': 0.15765186542626233, 'l1_ratio': 0.9994208008276525}


[I 2026-07-23 12:00:56,649] A new study created in memory with name: no-name-802e0b25-68ea-4491-a566-8a624d96305c
[I 2026-07-23 12:00:58,221] Trial 1 finished with value: 0.6928627141683432 and parameters: {'C': 0.002736920432364028, 'l1_ratio': 0.8710850015536578}. Best is trial 1 with value: 0.6928627141683432.
[I 2026-07-23 12:00:58,344] Trial 14 finished with value: 0.6928932762979538 and parameters: {'C': 0.0012741441497257904, 'l1_ratio': 0.5560047739237449}. Best is trial 1 with value: 0.6928627141683432.
[I 2026-07-23 12:00:58,508] Trial 3 finished with value: 0.6928449947966027 and parameters: {'C': 0.004172879238066467, 'l1_ratio': 0.9522567967293538}. Best is trial 3 with value: 0.6928449947966027.
[I 2026-07-23 12:01:11,373] Trial 15 finished with value: 0.6347347212362514 and parameters: {'C': 0.006758710648413022, 'l1_ratio': 0.22490023663349767}. Best is trial 15 with value: 0.6347347212362514.
[I 2026-07-23 12:01:11,925] Trial 2 finished with value: 0.6651136378451297 a

In [ ]:
window_results = []
all_val_true, all_val_proba = [], []
all_test_true, all_test_proba = [], []

for i, (split, params) in enumerate(zip(splits, parameters)):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel = selector.transform(X_val_scaled)
    X_test_sel = selector.transform(X_test_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_proba = model.predict_proba(X_val_sel)[:, 1]
    test_proba = model.predict_proba(X_test_sel)[:, 1]

    window_results.append({
        "window": i,
        "n_val": len(y_val),
        "n_test": len(y_test),
        "val_logloss": log_loss(y_val, val_proba),
        "test_logloss": log_loss(y_test, test_proba),
        "test_acc": accuracy_score(y_test, (test_proba > 0.5).astype(int)),
    })

    all_val_true.append(y_val); all_val_proba.append(val_proba)
    all_test_true.append(y_test); all_test_proba.append(test_proba)

# --- Pool everything: concatenate, don't average ---
# Concatenating before scoring naturally weights each window by its
# sample count. Window 6 (252 matches) counts 252x, Window 2 (32) counts
# 32x — no manual weighting needed, unlike averaging six per-window scores.

pooled_val_true = np.concatenate(all_val_true)
pooled_val_proba = np.concatenate(all_val_proba)
pooled_test_true = np.concatenate(all_test_true)
pooled_test_proba = np.concatenate(all_test_proba)

pooled_test_logloss = log_loss(pooled_test_true, pooled_test_proba)
pooled_test_brier = brier_score_loss(pooled_test_true, pooled_test_proba)
pooled_test_acc = accuracy_score(pooled_test_true, (pooled_test_proba > 0.5).astype(int))

print(f"Pooled test log-loss: {pooled_test_logloss:.4f}")
print(f"Pooled test Brier:    {pooled_test_brier:.4f}")
print(f"Pooled test accuracy: {pooled_test_acc:.4f}  (n={len(pooled_test_true)})")

for r in window_results:
    print(r)


# --- Bootstrap CI, so you can tell if an iteration actually moved the needle ---

def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi


mean_ll, lo, hi = bootstrap_logloss_ci(pooled_test_true, pooled_test_proba)
print(f"Pooled log-loss: {mean_ll:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Pooled test log-loss: 0.6571
Pooled test Brier:    0.2326
Pooled test accuracy: 0.5686  (n=452)
{'window': 0, 'n_val': 297, 'n_test': 102, 'val_logloss': 0.6301001584175471, 'test_logloss': 0.6702951837867613, 'test_acc': 0.5392156862745098}
{'window': 1, 'n_val': 198, 'n_test': 48, 'val_logloss': 0.6931065144779236, 'test_logloss': 0.6924686572455035, 'test_acc': 0.5625}
{'window': 2, 'n_val': 375, 'n_test': 72, 'val_logloss': 0.628807109654831, 'test_logloss': 0.6277623479715826, 'test_acc': 0.5972222222222222}
{'window': 3, 'n_val': 378, 'n_test': 102, 'val_logloss': 0.6131990832558103, 'test_logloss': 0.651114954597501, 'test_acc': 0.5490196078431373}
{'window': 4, 'n_val': 378, 'n_test': 72, 'val_logloss': 0.616747899574512, 'test_logloss': 0.6275313995013823, 'test_acc': 0.6388888888888888}
{'window': 5, 'n_val': 378, 'n_test': 56, 'val_logloss': 0.6443902713129145, 'test_logloss': 0.689546163694704, 'test_acc': 0.5357142857142857}
Pooled log-loss: 0.6574  95% CI [0.6293, 0.6894]

Pooled test log-loss: 0.6571
Pooled test Brier:    0.2326
Pooled test accuracy: 0.5686  (n=452)
{'window': 0, 'n_val': 297, 'n_test': 102, 'val_logloss': 0.6301001584175471, 'test_logloss': 0.6702951837867613, 'test_acc': 0.5392156862745098}
{'window': 1, 'n_val': 198, 'n_test': 48, 'val_logloss': 0.6931065144779236, 'test_logloss': 0.6924686572455035, 'test_acc': 0.5625}
{'window': 2, 'n_val': 375, 'n_test': 72, 'val_logloss': 0.628807109654831, 'test_logloss': 0.6277623479715826, 'test_acc': 0.5972222222222222}
{'window': 3, 'n_val': 378, 'n_test': 102, 'val_logloss': 0.6131990832558103, 'test_logloss': 0.651114954597501, 'test_acc': 0.5490196078431373}
{'window': 4, 'n_val': 378, 'n_test': 72, 'val_logloss': 0.616747899574512, 'test_logloss': 0.6275313995013823, 'test_acc': 0.6388888888888888}
{'window': 5, 'n_val': 378, 'n_test': 56, 'val_logloss': 0.6443902713129145, 'test_logloss': 0.689546163694704, 'test_acc': 0.5357142857142857}
Pooled log-loss: 0.6574  95% CI [0.6293, 0.6894]